# 07 — Building a Production Agent Tool Loop — Practical Examples

**Covers**: Complete ProductionAgent class, AgentState tracking, multi-condition termination, cost/step reporting

In [ ]:
import os, json, time, logging
from dataclasses import dataclass, field
from enum import Enum
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
logging.basicConfig(level=logging.WARNING)
print('✅ Ready')

## 1. AgentState — The Source of Truth

In [ ]:
class AgentStatus(Enum):
    RUNNING = 'running'
    COMPLETED = 'completed'
    FAILED = 'failed'
    MAX_STEPS = 'max_steps'

@dataclass
class AgentState:
    goal: str
    messages: list = field(default_factory=list)
    status: AgentStatus = AgentStatus.RUNNING
    step_count: int = 0
    max_steps: int = 10
    tool_calls_made: int = 0
    final_answer: str = ''
    input_tokens: int = 0
    output_tokens: int = 0
    start_time: float = field(default_factory=time.time)
    tool_log: list = field(default_factory=list)
    
    @property
    def elapsed(self): return round(time.time() - self.start_time, 2)
    
    @property
    def cost_usd(self):
        return (self.input_tokens * 0.00015 + self.output_tokens * 0.0006) / 1000
    
    def should_stop(self):
        return self.step_count >= self.max_steps or self.status != AgentStatus.RUNNING
    
    def report(self):
        t = Table(title='Agent Run Report')
        t.add_column('Metric', style='cyan')
        t.add_column('Value', style='green')
        t.add_row('Status', self.status.value)
        t.add_row('Steps Used', f'{self.step_count}/{self.max_steps}')
        t.add_row('Tool Calls', str(self.tool_calls_made))
        t.add_row('Input Tokens', str(self.input_tokens))
        t.add_row('Output Tokens', str(self.output_tokens))
        t.add_row('Cost (USD)', f'${self.cost_usd:.5f}')
        t.add_row('Time (s)', str(self.elapsed))
        rprint(t)

## 2. Production Agent Class

In [ ]:
class ProductionAgent:
    def __init__(self, system_prompt, tools, tool_map, max_steps=8):
        self.system_prompt = system_prompt
        self.tools = tools
        self.tool_map = tool_map
        self.max_steps = max_steps
    
    def run(self, user_input: str) -> AgentState:
        state = AgentState(goal=user_input, max_steps=self.max_steps)
        state.messages = [
            {'role': 'system', 'content': self.system_prompt},
            {'role': 'user', 'content': user_input}
        ]
        
        while not state.should_stop():
            state.step_count += 1
            
            # LLM call with basic retry
            response = self._call_llm(state, retries=2)
            if response is None:
                state.status = AgentStatus.FAILED
                state.final_answer = 'LLM call failed'
                break
            
            if response.usage:
                state.input_tokens  += response.usage.prompt_tokens
                state.output_tokens += response.usage.completion_tokens
            
            msg = response.choices[0].message
            state.messages.append(msg)
            finish = response.choices[0].finish_reason
            
            if finish == 'tool_calls':
                for tc in msg.tool_calls:
                    state.tool_calls_made += 1
                    name = tc.function.name
                    try:
                        args = json.loads(tc.function.arguments)
                    except:
                        args = {}
                    t0 = time.time()
                    result = self._safely_run(name, args)
                    dur = round(time.time() - t0, 3)
                    state.tool_log.append({'tool': name, 'args': args, 'result': result[:60], 'dur_s': dur})
                    print(f'  [{state.step_count}] {name}({args}) → {result[:50]} ({dur}s)')
                    state.messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
            else:
                state.final_answer = msg.content or ''
                state.status = AgentStatus.COMPLETED
                break
        
        if state.should_stop() and state.status == AgentStatus.RUNNING:
            state.status = AgentStatus.MAX_STEPS
            state.final_answer = f'Stopped at max {self.max_steps} steps.'
        
        return state
    
    def _call_llm(self, state, retries=2):
        for attempt in range(retries):
            try:
                return client.chat.completions.create(
                    model=MODEL, messages=state.messages,
                    tools=self.tools, tool_choice='auto',
                    temperature=0.1, max_tokens=1024
                )
            except Exception as e:
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)
        return None
    
    def _safely_run(self, name: str, args: dict) -> str:
        if name not in self.tool_map:
            return f'ERROR: tool "{name}" not found'
        try:
            r = self.tool_map[name](**args)
            return str(r)[:3000]
        except Exception as e:
            return f'ERROR: {e}'

## 3. Demo — Run the Agent

In [ ]:
TOOLS = [
  {'type':'function','function':{'name':'web_search','description':'Search web for current info.','parameters':{'type':'object','properties':{'query':{'type':'string'}},'required':['query']}}},
  {'type':'function','function':{'name':'calculator','description':'Evaluate math expression.','parameters':{'type':'object','properties':{'expression':{'type':'string'}},'required':['expression']}}}
]

def web_search(query: str) -> str:
    db = {'python': 'Python 3.13 released Oct 2024', 'openai': 'GPT-4o released May 2024'}
    for k, v in db.items():
        if k in query.lower(): return v
    return f'No results for: {query}'

def calculator(expression: str) -> str:
    try: return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e: return f'Math error: {e}'

SYSTEM = 'You are a research assistant. Use tools. When done, answer directly.'
agent = ProductionAgent(SYSTEM, TOOLS, {'web_search': web_search, 'calculator': calculator}, max_steps=6)

state = agent.run('What is the latest Python version and what is 18% of 4200?')

print(f'\nFINAL ANSWER:\n{state.final_answer}')
print()
state.report()

## 4. Multi-Condition Termination Tests

In [ ]:
# Test max_steps guard
agent_tiny = ProductionAgent(SYSTEM, TOOLS, {'web_search': web_search, 'calculator': calculator}, max_steps=1)
state = agent_tiny.run('Research Python, then search OpenAI, then calculate 5+5')
print(f'Status (should be max_steps): {state.status.value}')
print(f'Steps used: {state.step_count}')